In [1]:
import pandas as pd
import duckdb
print("Environnement prêt", pd.__version__, duckdb.__version__)

Environnement prêt 3.0.6 1.5.6


## 1. Chargement du fichier

In [2]:
df = pd.read_csv("../data/BankChurners.csv")
print(df.shape)

(10127, 23)


### Dictionnaire des données

| Colonne | Signification |
|---|---|
| CLIENTNUM | Identifiant unique du client |
| Attrition_Flag | Statut du client : « Existing Customer » = actif, « Attrited Customer » = parti |
| Customer_Age | Âge du client |
| Gender | Sexe (F / M) |
| Dependent_count | Nombre de personnes à charge |
| Education_Level | Niveau d'études |
| Marital_Status | Situation familiale |
| Income_Category | Tranche de revenu annuel (en dollars) |
| Card_Category | Gamme de carte : Blue, Silver, Gold, Platinum |
| Months_on_book | Ancienneté du client (en mois) |
| Total_Relationship_Count | Nombre de produits détenus auprès de la banque |
| Months_Inactive_12_mon | Nombre de mois sans activité sur les 12 derniers mois |
| Contacts_Count_12_mon | Nombre de contacts avec la banque sur les 12 derniers mois |
| Credit_Limit | Plafond de la carte |
| Total_Revolving_Bal | Encours renouvelable : montant utilisé et non encore remboursé |
| Avg_Open_To_Buy | Montant encore disponible sur la carte (moyenne sur 12 mois) |
| Total_Amt_Chng_Q4_Q1 | Évolution du montant des transactions entre le 1er et le 4e trimestre |
| Total_Trans_Amt | Montant total des transactions sur 12 mois |
| Total_Trans_Ct | Nombre total de transactions sur 12 mois |
| Total_Ct_Chng_Q4_Q1 | Évolution du nombre de transactions entre le 1er et le 4e trimestre |
| Avg_Utilization_Ratio | Taux moyen d'utilisation du plafond (encours ÷ plafond) |

*Les 2 colonnes « Naive_Bayes_Classifier… » sont traitées en section 2.*

## 2. Examen et suppression des 2 dernières colonnes

In [3]:
# Raccourcir les deux noms trop longs pour les lire facilement
col1, col2 = df.columns[-2], df.columns[-1]

# Indice 2 : à quoi ressemblent les valeurs ?
print(df[[col1, col2]].head())

# Indice 3 : la somme des deux colonnes vaut-elle toujours 1 ?
print((df[col1] + df[col2]).round(3).value_counts())

# Indice 4 : la colonne 1 correspond-elle aux clients partis ?
print(df.groupby("Attrition_Flag")[col1].mean().round(3))

   Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1  \
0                                           0.000093                                                                                    
1                                           0.000057                                                                                    
2                                           0.000021                                                                                    
3                                           0.000134                                                                                    
4                                           0.000022                                                                                    

   Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2  
0                                        

**Constat**
- Les 2 dernières colonnes portent le nom d'un modèle de prédiction (« Naive_Bayes_Classifier ») associé à « Attrition_Flag », la variable étudiée.
- Leurs valeurs sont des probabilités comprises entre 0 et 1, dont la somme est égale à 1 pour l'ensemble des 10 127 clients.
- La première colonne atteint en moyenne 0,995 pour les clients partis et 0,000 pour les clients restés : elle reproduit le résultat de manière quasi parfaite.
- La documentation de la source confirme qu'il s'agit de calculs réalisés par l'auteur, à exclure de l'analyse.

**Décision** : exclusion de ces 2 colonnes avant toute analyse.

**Enjeu métier**
- Ces deux variables sont issues d'un modèle de prédiction calculé par l'auteur du jeu de données : elles reflètent le résultat (départ ou maintien) et non le profil ou le comportement du client. Les conserver conduirait à des conclusions artificiellement parfaites, inexploitables en situation réelle.
- L'objectif de l'analyse est d'identifier des signaux observables avant le départ (baisse d'utilisation de la carte, inactivité, sollicitations du service client), sur lesquels les équipes de relation client peuvent agir. Seules les informations dont la banque dispose au moment de décider sont donc retenues.
- Les signaux d'alerte présentés dans la suite de l'analyse pourront ainsi être suivis par la banque dans ses outils actuels, à partir des données dont elle dispose déjà.

In [4]:
df = df.drop(columns=[col1, col2])
print(df.shape)

(10127, 21)


## 3. Répartition clients restés / partis

In [6]:
# Attrition_Flag : "Existing Customer" = client actif, "Attrited Customer" = client parti
print(df["Attrition_Flag"].value_counts())
print(df["Attrition_Flag"].value_counts(normalize=True).round(3))

Attrition_Flag
Existing Customer    8500
Attrited Customer    1627
Name: count, dtype: int64
Attrition_Flag
Existing Customer    0.839
Attrited Customer    0.161
Name: proportion, dtype: float64


**Constat**
- 1 627 clients sur 10 127 ont quitté la banque, soit un taux de départ de 16,1 % (environ 1 client sur 6).
- 8 500 clients sont restés actifs (83,9 %).

**Enjeu métier**
- Le taux de 16,1 % constitue la référence de l'analyse : chaque groupe de clients sera comparé à ce taux pour mesurer son niveau de risque. Un groupe affichant 30 % de départs présente un risque près de 2 fois supérieur à la moyenne.
- Un tel niveau de départs représente une perte significative de revenus récurrents (cotisations, commissions, intérêts) et augmente les coûts de conquête nécessaires pour maintenir le portefeuille. Fidéliser un client existant coûte généralement moins cher que d'en acquérir un nouveau.
- Les clients partis étant minoritaires, les comparaisons reposeront sur des taux de départ par groupe plutôt que sur des volumes, afin de ne pas surévaluer le poids des clients restés.

## 4. Contrôle des doublons et des valeurs manquantes

In [7]:
# CLIENTNUM : identifiant unique du client
print("Lignes en double :", df.duplicated().sum())
print("Identifiants clients en double :", df["CLIENTNUM"].duplicated().sum())
print("Valeurs manquantes :", df.isna().sum().sum())

Lignes en double : 0
Identifiants clients en double : 0
Valeurs manquantes : 0


**Constat**
- Aucune ligne en double et aucun identifiant client répété : chaque ligne correspond à un client unique.
- Aucune valeur manquante détectée au sens strict (cellule vide).

**Décision** : aucune correction nécessaire à ce stade. Le contrôle des valeurs renseignées « Unknown », qui ne sont pas détectées comme manquantes, fait l'objet de la section suivante.

**Enjeu métier**
- La base recense 10 127 clients distincts : les chiffres présentés reflètent fidèlement le portefeuille réel, sans client compté deux fois.
- Le taux de départ et la taille de chaque groupe de clients peuvent donc être utilisés tels quels pour dimensionner les actions de fidélisation (nombre de clients à contacter, budget à prévoir).

## 5. Valeurs renseignées « Unknown »

In [8]:
# Education_Level : niveau d'études / Marital_Status : situation familiale / Income_Category : tranche de revenu
# "Unknown" = information inconnue
colonnes = ["Education_Level", "Marital_Status", "Income_Category"]

# Nombre et part de clients avec une information inconnue, par colonne
for col in colonnes:
    nb = (df[col] == "Unknown").sum()
    print(f"{col} : {nb} clients ({nb / len(df):.1%})")

# Clients ayant au moins une information inconnue
inconnu = (df[colonnes] == "Unknown").any(axis=1)
print("Au moins une information inconnue :", inconnu.sum(), f"({inconnu.mean():.1%})")

# Taux de départ : clients avec information inconnue (True) vs complète (False)
parti = df["Attrition_Flag"] == "Attrited Customer"
print(parti.groupby(inconnu).mean().round(3))

Education_Level : 1519 clients (15.0%)
Marital_Status : 749 clients (7.4%)
Income_Category : 1112 clients (11.0%)
Au moins une information inconnue : 3046 (30.1%)
False    0.157
True     0.169
Name: Attrition_Flag, dtype: float64


**Constat**
- Certaines informations du profil client sont renseignées « Unknown » (inconnu) : niveau d'études pour 15,0 % des clients, tranche de revenu pour 11,0 %, situation familiale pour 7,4 %.
- Au total, 3 046 clients (30,1 %) ont au moins une information inconnue.
- Leur taux de départ (16,9 %) est proche de celui des clients au profil complet (15,7 %).

**Décision** : conservation de ces clients, avec « Inconnu » traité comme une catégorie à part entière.

**Enjeu métier**
- Écarter ces clients reviendrait à ignorer près d'un tiers du portefeuille, dont environ 500 clients partis. Les actions de fidélisation passeraient à côté d'une part importante des clients à risque.
- Ces clients se comportent comme les autres : leur présence ne fausse pas les résultats.
- Près d'un client sur trois a une fiche incomplète. Compléter ces informations lors des prochains contacts (agence, service client) permettrait à la banque de mieux cibler ses offres et ses actions de fidélisation.

## 6. Valeurs extrêmes et cohérence des données

In [12]:
# Valeurs minimales, médianes et maximales des colonnes chiffrées
# (traductions : voir le dictionnaire des données en section 1)
# CLIENTNUM est exclu : c'est un identifiant, pas une mesure
with pd.option_context("display.float_format", "{:.2f}".format):
    display(df.drop(columns="CLIENTNUM").describe().T[["min", "50%", "max"]])

,min,50%,max
Customer_Age,26.00,46.00,73.00
Dependent_count,0.00,2.00,5.00
Months_on_book,13.00,36.00,56.00
Total_Relationship_Count,1.00,4.00,6.00
Months_Inactive_12_mon,0.00,2.00,6.00
Contacts_Count_12_mon,0.00,2.00,6.00
Credit_Limit,1438.30,4549.00,34516.00
Total_Revolving_Bal,0.00,1276.00,2517.00
Avg_Open_To_Buy,3.00,3474.00,34516.00
Total_Amt_Chng_Q4_Q1,0.00,0.74,3.40


In [10]:
# Credit_Limit : plafond / Total_Revolving_Bal : encours / Avg_Open_To_Buy : montant disponible
# Contrôle : plafond = encours + montant disponible
ecart = (df["Credit_Limit"] - df["Total_Revolving_Bal"] - df["Avg_Open_To_Buy"]).abs()
print("Clients pour lesquels plafond ≠ encours + disponible :", (ecart > 1).sum())

# Total_Revolving_Bal : clients sans encours
print("Clients sans encours :", (df["Total_Revolving_Bal"] == 0).sum())

# Months_on_book : ancienneté en mois, répartition autour de 36 mois
print(df["Months_on_book"].value_counts().sort_index().loc[33:39])

Clients pour lesquels plafond ≠ encours + disponible : 0
Clients sans encours : 2470
Months_on_book
33     305
34     353
35     317
36    2463
37     358
38     347
39     341
Name: count, dtype: int64


**Constat**
- Les valeurs observées sont plausibles pour une clientèle de cartes de crédit : âge de 26 à 73 ans, ancienneté de 13 à 56 mois, plafond de 1 438 à 34 516 $. Aucune valeur aberrante (âge ou montant négatif, plafond nul) n'est détectée.
- Le plafond est égal à la somme de l'encours et du montant disponible pour 100 % des clients : les données financières sont cohérentes entre elles.
- 2 470 clients (24,4 %) n'ont aucun encours : ils remboursent la totalité de leurs dépenses ou utilisent peu leur carte.
- 2 463 clients (24,3 %) affichent exactement 36 mois d'ancienneté, contre 300 à 360 clients pour les mois voisins. Ce pic suggère une valeur attribuée par défaut lorsque l'ancienneté réelle était inconnue.

**Décision** : conservation de l'ensemble des clients. L'ancienneté sera interprétée avec prudence et ne servira pas de critère principal de ciblage.

**Enjeu métier**
- Les montants (plafond, encours, transactions) sont fiables et peuvent être utilisés directement pour repérer les clients dont l'activité baisse.
- L'ancienneté d'environ un quart des clients n'est pas fiable : elle ne sera pas retenue comme critère de ciblage. Les actions de fidélisation s'appuieront en priorité sur l'utilisation de la carte, une information fiable et suivie en continu.
- Fiabiliser la date d'entrée en relation de ces clients dans les outils de la banque permettrait d'intégrer ce critère aux prochaines analyses.